# Raw Transcript Inspection & Reformatting

Transcripts that did **not** come from WhisperX (platform-scraped or caption-derived) can show
up in all kinds of inconsistent layouts — different timestamp formats, different ways of marking
who's speaking, sometimes both varying file-to-file. WhisperX output, by contrast, is already
consistent: `[MM:SS] SPEAKER: text`, one turn per line.

This notebook's job is to bring the *non*-WhisperX transcripts in `raw_transcripts/` up to that
same standard before they go into `transcripts/`, where the rest of the pipeline expects
everything to look the same regardless of where it originally came from.

**Workflow:**
1. Inspect every raw file - print a sample and run some lightweight pattern-detection to guess
   its timestamp style and speaker style.
2. Based on what you see, assign each file to a **format group** (reuse a group across files that
   share a layout; give a file its own entry if it's genuinely unique).
3. Write one fix function per format group, each converting that layout's lines into
   `[MM:SS] SPEAKER: text`.
4. Apply the right fix function to each file and save the result to `transcripts/`.
5. Audit the output to catch anything that didn't come out clean.


## 0 · Configuration

In [30]:
import re
from pathlib import Path

RAW_DIR = Path("../raw_transcripts")   # input: one .txt per documentary, mixed formats
OUT_DIR = Path("../transcripts")       # output: same filenames, all in [MM:SS] SPEAKER: text

OUT_DIR.mkdir(parents=True, exist_ok=True)

raw_files = sorted(p for p in RAW_DIR.glob("*.txt") if p.is_file())
print(f"{len(raw_files)} raw transcript(s) found in {RAW_DIR.resolve()}")
for p in raw_files:
    print(" -", p.name)

13 raw transcript(s) found in /sfs/weka/scratch/pkb5nz/raw_transcripts
 - 91%- A Film About Guns in America_Transcript.txt
 - American Tragedy_Transcript.txt
 - Charm City_Transcript.txt
 - GunnedDown.txt
 - Living for 32_Transcript.txt
 - Newtown.txt
 - Romeo Is Bleeding_Transcript.txt
 - The Armor of Light_Transcript.txt
 - The Brutal Truth- A Violence Documentary_Transcript.txt
 - TheInterrupters.txt
 - ToughGuise2.txt
 - Under the Gun_Transcript.txt
 - When the Shooting Stops_Transcript.txt


## 1 · Inspect each raw transcript

For every file: print a sample of lines, then diff its actual structure against the WhisperX
target shape (`[MM:SS] SPEAKER: text`) - which timestamp style(s) it uses, whether the
timestamp is glued directly to the text, whether a `SPEAKER:` label is present at all (and
whether *that* is glued to its text), and whether any line already matches the target shape
outright. Files using a plain `MM:SS` style throughout are also checked for an hour rollover -
the ambiguous case where a timestamp like `59:59` is followed by `01:00`, which could mean
"1 hour, 0 minutes" or genuinely "1 minute" with no way to tell from the text alone.

In [31]:
# ── Timestamp classification: mutually exclusive, most-specific-first ──────
TS_PATTERNS = [
    ("bracket_hh_mm_ss", re.compile(r"^\[(\d{2}):(\d{2}):(\d{2})\]")),
    ("bracket_mm_ss",    re.compile(r"^\[(\d{1,2}):(\d{2})\]")),
    ("bare_hh_mm_ss",    re.compile(r"^(\d{2}):(\d{2}):(\d{2})")),
    ("bare_mm_ss",       re.compile(r"^(\d{1,2}):(\d{2})\b")),
]

def classify_line(line):
    for style, pat in TS_PATTERNS:
        m = pat.match(line)
        if m:
            return style, m.groups(), line[m.end():]
    return "none", None, line

def spaced_examples(indices, k=3):
    if not indices:
        return []
    if len(indices) <= k:
        return indices
    positions = [round(f * (len(indices) - 1)) for f in (0.1, 0.5, 0.9)][:k]
    seen, out = set(), []
    for pos in positions:
        idx = indices[pos]
        if idx not in seen:
            seen.add(idx)
            out.append(idx)
    return out

def timestamp_report(path):
    raw_lines = [l for l in path.read_text(encoding="utf-8", errors="ignore").splitlines() if l.strip()]
    n = len(raw_lines)

    styles = []
    by_style = {}   # style -> list of line indices
    glued_ts = 0
    rollover_flags = []
    prev_val = None

    for i, line in enumerate(raw_lines):
        style, groups, rest = classify_line(line)
        styles.append(style)
        by_style.setdefault(style, []).append(i)

        if style != "none":
            if rest and not rest.startswith(" "):
                glued_ts += 1
            if style in ("bracket_mm_ss", "bare_mm_ss"):
                mm, ss = int(groups[0]), int(groups[1])
                val = mm * 60 + ss
                if prev_val is not None and val < prev_val - 5:
                    rollover_flags.append((i, prev_val, val))
                prev_val = val

    style_counts = {s: styles.count(s) for s in set(styles)}
    # dominant style = the one with the most matches (excluding 'none')
    dominant = max((s for s in style_counts if s != "none"), key=lambda s: style_counts[s], default=None)

    print("=" * 100)
    print(f"{path.name}  ({n} non-empty lines)")
    print("-" * 100)

    print("Timestamp style:", style_counts,
          " <-- mixed styles, worth a closer look" if len(style_counts) > 1 else "")
    if glued_ts:
        print(f"  (timestamp glued directly to text, no space, in {glued_ts} line(s))")

    if dominant:
        ex = spaced_examples(by_style[dominant], k=1)
        print("  Example line of style:")
        for idx in ex:
            print(f"    [{idx}] {raw_lines[idx][:110]}")

    other_styles = [s for s in style_counts if s != dominant]
    for s in other_styles:
        ex = spaced_examples(by_style[s], k=1)
        print(f"  Example line of other style:")
        for idx in ex:
            print(f"    [{idx}] {raw_lines[idx][:110]}")
    print()

    if rollover_flags:
        print(f"Hour rollover: POSSIBLE at {len(rollover_flags)} point(s) (MM:SS value drops unexpectedly):")
        for i, prev_val, val in rollover_flags[:5]:
            print(f"    line {i}: {prev_val//60:02d}:{prev_val%60:02d} -> {val//60:02d}:{val%60:02d}"
                  f"  (raw: \"{raw_lines[i][:60]}\")")
    else:
        has_mm_ss_only = any(s in ("bracket_mm_ss", "bare_mm_ss") for s in style_counts)
        if has_mm_ss_only:
            print("Hour rollover: none detected (confirm the film is under 1hr, or check manually")
            print("near the runtime's 59:xx mark - a short gap there can mask a rollover).")
        else:
            print("Hour rollover: n/a (timestamps aren't a plain MM:SS style)")
    print()

    signature = (tuple(sorted(style_counts.keys())), bool(rollover_flags))
    return signature, path.name

ts_signatures = [timestamp_report(p) for p in raw_files]

# ── Cross-file timestamp-format matches ─────────────────────────────────────
print("=" * 100)
print("TIMESTAMP FORMAT GROUPS ACROSS FILES")
print("-" * 100)
ts_groups = {}
for sig, name in ts_signatures:
    ts_groups.setdefault(sig, []).append(name)

for sig, names in ts_groups.items():
    styles, has_rollover = sig
    rollover_note = " + HOUR ROLLOVER" if has_rollover else ""
    print(f"Timestamp style {styles}{rollover_note}  ->  {len(names)} file(s):")
    for name in names:
        print(f"    - {name}")
    print()

if not matched_any:
    print("No two files share the exact same timestamp format.")

91%- A Film About Guns in America_Transcript.txt  (362 non-empty lines)
----------------------------------------------------------------------------------------------------
Timestamp style: {'bare_hh_mm_ss': 359, 'none': 3}  <-- mixed styles, worth a closer look
  (timestamp glued directly to text, no space, in 359 line(s))
  Example line of style:
    [39] 00:03:45Prohibits interstate firearms transfers, except among licensed manufacturers, dealers and importers. 
  Example line of other style:
    [0] 91%: A Film About Guns in America. Directed by John Richie. Films Media Group, 2016. Alexander Street.

Hour rollover: n/a (timestamps aren't a plain MM:SS style)

American Tragedy_Transcript.txt  (1354 non-empty lines)
----------------------------------------------------------------------------------------------------
Timestamp style: {'bare_hh_mm_ss': 1351, 'none': 3}  <-- mixed styles, worth a closer look
  (timestamp glued directly to text, no space, in 1351 line(s))
  Example line 

In [32]:
import re
from pathlib import Path

FILES = [
    "91%- A Film About Guns in America_Transcript.txt",
    "American Tragedy_Transcript.txt",
    "Charm City_Transcript.txt",
    "Living for 32_Transcript.txt",
    "Romeo Is Bleeding_Transcript.txt",
    "The Armor of Light_Transcript.txt",
    "The Brutal Truth- A Violence Documentary_Transcript.txt",
    "Under the Gun_Transcript.txt",
    "When the Shooting Stops_Transcript.txt",
]

RAW_DIR = Path("../raw_transcripts")
OUT_DIR = Path("../raw_transcripts/in_progress")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Format: 00:24:50SPEAKER: text  or  00:24:50some text (no speaker)
# Convert HH:MM:SS -> [MM:SS] preserving everything after the timestamp as-is
TS_RE = re.compile(r"^(\d{2}):(\d{2}):(\d{2})(.*)")

def fix_bare_hh_mm_ss(path):
    raw_lines = [l for l in path.read_text(encoding="utf-8", errors="ignore").splitlines() if l.strip()]
    out = []
    for line in raw_lines:
        m = TS_RE.match(line)
        if not m:
            continue   # drop header/metadata lines with no timestamp (the 'none' style lines)
        hh, mm, ss, rest = int(m.group(1)), int(m.group(2)), int(m.group(3)), m.group(4)
        total_mm = hh * 60 + mm
        ts = f"[{total_mm:02d}:{ss:02d}]"
        rest = rest.strip()
        if rest:
            out.append(f"{ts} {rest}")
    return out

for fname in FILES:
    path = RAW_DIR / fname
    lines = fix_bare_hh_mm_ss(path)
    out_path = OUT_DIR / fname
    out_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    print(f"{fname}: {len(lines)} lines -> {out_path.name}")

91%- A Film About Guns in America_Transcript.txt: 359 lines -> 91%- A Film About Guns in America_Transcript.txt
American Tragedy_Transcript.txt: 1351 lines -> American Tragedy_Transcript.txt
Charm City_Transcript.txt: 1738 lines -> Charm City_Transcript.txt
Living for 32_Transcript.txt: 218 lines -> Living for 32_Transcript.txt
Romeo Is Bleeding_Transcript.txt: 1817 lines -> Romeo Is Bleeding_Transcript.txt
The Armor of Light_Transcript.txt: 1502 lines -> The Armor of Light_Transcript.txt
The Brutal Truth- A Violence Documentary_Transcript.txt: 0 lines -> The Brutal Truth- A Violence Documentary_Transcript.txt
Under the Gun_Transcript.txt: 1915 lines -> Under the Gun_Transcript.txt
When the Shooting Stops_Transcript.txt: 157 lines -> When the Shooting Stops_Transcript.txt
